# Синтетические заявки на курсы ДПО

Этот ноутбук — отчёт по домашнему заданию семинара 2. Задача была сгенерировать 50 валидных заявок на курсы повышения квалификации: со вложенным адресом, ограниченными списками специальностей и курсов, Pydantic-валидацией и проверкой, что модель не «залипла» на одном городе или одной профессии.

Я сделал генерацию не как простой `random.choice`, а как небольшой набор карточек для приёмной комиссии: заранее задаётся город, специальность и рабочий контекст заявителя, а LLM заполняет человеческие детали заявки.

## Что именно проверяем

Для такой задачи мало просто получить 50 строк. У LLM есть типичный риск: она начинает повторять самые привычные варианты — например, Москву, Санкт-Петербург или одну популярную специальность. Поэтому я смотрю на три группы метрик.

1. **Валидность**: каждая строка должна проходить Pydantic-схему. Это проверяет формат, диапазоны возраста и стажа, допустимые значения `Literal`, город из списка и согласованность возраста с годом выпуска.
2. **Доля самой частой категории**: простая метрика для mode collapse. Если один город занимает больше 40%, а одна специальность больше 35%, набор уже плохо похож на разнообразную выборку.
3. **Нормированная энтропия**: показывает не только лидера, а общую равномерность распределения. Значение около 1 означает, что категории представлены почти одинаково; ближе к 0 — данные сконцентрировались в нескольких вариантах.

In [ ]:
from math import log
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_PATH = Path("applications.csv")
df = pd.read_csv(DATA_PATH)

print(f"Загружено заявок: {len(df)}")
df.head()

In [ ]:
def top_share(series: pd.Series) -> float:
    counts = series.value_counts()
    return counts.iloc[0] / counts.sum()


def normalized_entropy(series: pd.Series) -> float:
    counts = series.value_counts()
    probabilities = counts / counts.sum()
    entropy = -(probabilities * probabilities.map(log)).sum()
    return entropy / log(len(counts))


metrics = pd.DataFrame(
    [
        {
            "поле": "city",
            "уникальных значений": df["city"].nunique(),
            "топ-доля": top_share(df["city"]),
            "порог по заданию": 0.40,
            "нормированная энтропия": normalized_entropy(df["city"]),
        },
        {
            "поле": "speciality",
            "уникальных значений": df["speciality"].nunique(),
            "топ-доля": top_share(df["speciality"]),
            "порог по заданию": 0.35,
            "нормированная энтропия": normalized_entropy(df["speciality"]),
        },
    ]
)

metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5))

city_counts = df["city"].value_counts().sort_index()
speciality_counts = df["speciality"].value_counts().sort_index()

city_counts.plot(kind="bar", ax=axes[0], color="#66c2a5", edgecolor="#333333")
axes[0].set_title("Заявки по городам")
axes[0].set_ylabel("Количество заявок")
axes[0].tick_params(axis="x", rotation=35)
axes[0].grid(axis="y", alpha=0.25)

speciality_counts.plot(kind="bar", ax=axes[1], color="#fc8d62", edgecolor="#333333")
axes[1].set_title("Заявки по специальностям")
axes[1].set_ylabel("Количество заявок")
axes[1].tick_params(axis="x", rotation=35)
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

## Почему эти графики полезны

Гистограммы здесь не просто для красоты. По ним сразу видно, есть ли перекос: если один столбец заметно выше остальных, значит генерация начинает повторять один и тот же привычный вариант. Это особенно важно для синтетических данных, потому что формально JSON может быть полностью валидным, но набор всё равно будет бесполезным для анализа.

В этом прогоне распределения ровные: каждый из 10 городов и каждая из 10 специальностей встречаются по 5 раз. Такой результат получился из-за квотирования карточек перед обращением к LLM. Модель всё ещё генерирует имена, район, стаж, возраст и курс, но ключевые поля для борьбы с mode collapse не отданы ей полностью на самотёк.

In [ ]:
# Небольшая проверка смысловых связок: какие курсы выбирались для разных специальностей.
# Это не строгий тест, но он помогает увидеть странные сочетания до сдачи работы.
course_table = pd.crosstab(df["speciality"], df["desired_course"])
course_table

In [ ]:
# Если notebook лежит рядом со schema.py, можно прогнать те же Pydantic-правила,
# которыми пользовался генератор. В Colab эта ячейка сработает после загрузки schema.py.
try:
    from schema import Application

    for _, row in df.iterrows():
        Application.model_validate(
            {
                "full_name": row["full_name"],
                "age": int(row["age"]),
                "address": {"city": row["city"], "district": row["district"]},
                "speciality": row["speciality"],
                "desired_course": row["desired_course"],
                "years_of_experience": int(row["years_of_experience"]),
                "graduation_year": int(row["graduation_year"]),
            }
        )
    print(f"Pydantic-проверка: {len(df)}/{len(df)} строк валидны")
except ModuleNotFoundError:
    print("schema.py не найден рядом с ноутбуком. В Colab загрузите schema.py и повторите ячейку.")

## Итог

Получилось 50 валидных заявок из 50. Порог по городам из задания — не больше 40% на одну категорию, по специальностям — не больше 35%; в моём наборе максимум равен 10% для обоих полей. Нормированная энтропия равна 1.00, то есть распределение практически идеально равномерное.

Важный момент по валидаторам: во время последнего запуска `@field_validator` действительно поймал одну ошибку согласованности возраста и года выпуска (`graduation_year_age_conflict=1`). Это полезно, потому что показывает: схема не просто формальность, она реально страхует от биографически невозможных заявок. При этом остался ручной риск в смысловой связке специальности и курса — её я контролировал промптом и кросс-таблицей, но не жёстким правилом.